In [0]:
df_silver = spark.read.table("workspace.default.silver_nyctaxi")

In [0]:
df_gold = df_silver

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date, sum as _sum, avg, count

df_gold = (df_silver
    .withColumn("pickup_date", to_date(col("pickup_ts")))
    .groupBy("pickup_date")
    .agg(
        count("*").alias("total_corridas"),
        _sum("fare_amount").alias("faturamento_total"),
        avg("trip_distance").alias("distancia_media")
    )
    .orderBy(col("pickup_date").asc())
    .withColumnRenamed("pickup_date", "Data")
    .withColumn("Distancia_total", F.round(F.col("total_corridas") * F.col("distancia_media"), 2)
    )
    .withColumn("distancia_media", F.round(F.col("distancia_media"), 2)
    )
)

display(df_gold)

In [0]:
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_faturamento_diario"))

print("Gold gravado com sucesso.")

In [0]:
spark.sql("SELECT COUNT(*) AS total_linhas FROM workspace.default.silver_nyctaxi").show()

In [0]:
display(spark.sql("SHOW CATALOGS"))
display(spark.sql("SHOW SCHEMAS IN workspace"))
display(spark.sql("SHOW TABLES IN workspace.default"))